# Part A: Dataset Preparation

In [1]:
import os
import random
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from google.colab import drive

drive.mount('/content/drive')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("Random seed set to 42.")

data_dir = '/content/drive/MyDrive/MalaysiaFood'

#Data augmentation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Data loading & splitting
full_train_dataset = ImageFolder(root=data_dir, transform=train_transforms)
full_val_dataset = ImageFolder(root=data_dir, transform=test_transforms)

num_samples = len(full_train_dataset)
train_size = int(0.8 * num_samples)

indices = torch.randperm(num_samples).tolist()

train_dataset = Subset(full_train_dataset, indices[:train_size])
val_dataset = Subset(full_val_dataset, indices[train_size:])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Total images found: {num_samples}")
print(f"Classes: {full_train_dataset.classes}")
print(f"Training set size: {len(train_dataset)} images")
print(f"Validation set size: {len(val_dataset)} images")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Random seed set to 42.
Total images found: 3000
Classes: ['kaya_toast', 'laksa', 'nasi_lemak', 'popiah', 'roti_canai', 'satay']
Training set size: 2400 images
Validation set size: 600 images


# Part B: CNN Architecture Design

In [2]:
import torch.nn as nn
import torch.nn.functional as F

class MalaysianFoodCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(MalaysianFoodCNN, self).__init__()

        #Block 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1, stride=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        #Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        #Block 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1, stride=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        #Block 4
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1, stride=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # After 4 max-pools (224/2/2/2/2=14), spatial size is 14x14
        self.fc1 = nn.Linear(256 * 14 * 14, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = self.pool4(F.relu(self.conv4(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MalaysianFoodCNN(num_classes=6).to(device)
print(f"Model initialized on {device}")

from torchsummary import summary
summary(model, input_size=(3, 224, 224))

Model initialized on cpu
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
         MaxPool2d-2         [-1, 32, 112, 112]               0
            Conv2d-3         [-1, 64, 112, 112]          18,496
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5          [-1, 128, 56, 56]          73,856
         MaxPool2d-6          [-1, 128, 28, 28]               0
            Conv2d-7          [-1, 256, 28, 28]         295,168
         MaxPool2d-8          [-1, 256, 14, 14]               0
            Linear-9                  [-1, 512]      25,690,624
          Dropout-10                  [-1, 512]               0
           Linear-11                    [-1, 6]           3,078
Total params: 26,082,118
Trainable params: 26,082,118
Non-trainable params: 0
----------------------------------------------------------------

# Part C: Transfer Learning Comparison

In [3]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import models

weights = models.ResNet50_Weights.DEFAULT
resnet50 = models.resnet50(weights=weights)

for param in resnet50.parameters():
    param.requires_grad = False

for param in resnet50.layer4.parameters():
    param.requires_grad = True

num_ftrs = resnet50.fc.in_features
resnet50.fc = nn.Linear(num_ftrs, 6)

resnet50 = resnet50.to(device)
print("ResNet-50 initialized and modified for 6 classes.")

#Training engine
def train_model(model, dataloaders, criterion, optimizer, num_epochs=10):
    since = time.time()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'epoch_times': []}

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        epoch_start = time.time()
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = dataloaders['train']
            else:
                model.eval()
                dataloader = dataloaders['val']

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Save history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        epoch_time = time.time() - epoch_start
        history['epoch_times'].append(epoch_time)
        print(f'Epoch Time: {epoch_time:.0f}s\n')

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history, time_elapsed

ResNet-50 initialized and modified for 6 classes.


In [4]:
dataloaders = {'train': train_loader, 'val': val_loader}
criterion = nn.CrossEntropyLoss()

#Train Custom CNN
print("Training Custom CNN")
custom_optimizer = optim.Adam(model.parameters(), lr=0.001)
best_custom_model, custom_history, custom_total_time = train_model(
    model, dataloaders, criterion, custom_optimizer, num_epochs=10
)

#Train ResNet-50
print("\nTraining ResNet-50")
resnet_optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet50.parameters()), lr=0.0001)
best_resnet_model, resnet_history, resnet_total_time = train_model(
    resnet50, dataloaders, criterion, resnet_optimizer, num_epochs=10
)

#Plotting learning curves
epochs = range(1, 11)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, custom_history['train_acc'], 'b--', label='Custom CNN Train')
plt.plot(epochs, custom_history['val_acc'], 'b-', linewidth=2, label='Custom CNN Val')
plt.plot(epochs, resnet_history['train_acc'], 'r--', label='ResNet-50 Train')
plt.plot(epochs, resnet_history['val_acc'], 'r-', linewidth=2, label='ResNet-50 Val')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, custom_history['train_loss'], 'b--', label='Custom CNN Train')
plt.plot(epochs, custom_history['val_loss'], 'b-', linewidth=2, label='Custom CNN Val')
plt.plot(epochs, resnet_history['train_loss'], 'r--', label='ResNet-50 Train')
plt.plot(epochs, resnet_history['val_loss'], 'r-', linewidth=2, label='ResNet-50 Val')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Training Custom CNN
Epoch 1/10
----------
Train Loss: 1.7008 Acc: 0.2796
Val Loss: 1.5748 Acc: 0.3800
Epoch Time: 627s

Epoch 2/10
----------
Train Loss: 1.5165 Acc: 0.3900
Val Loss: 1.6292 Acc: 0.3750
Epoch Time: 524s

Epoch 3/10
----------


KeyboardInterrupt: 

# Part D: Analysis & Reflection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model_to_test = best_resnet_model
model_to_test.eval()

misclassified_images = []
misclassified_true = []
misclassified_preds = []

class_names = full_train_dataset.classes


with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model_to_test(inputs)
        _, preds = torch.max(outputs, 1)

        #Find the indices where the prediction does not match the actual label
        wrong_idx = (preds != labels).nonzero(as_tuple=True)[0]

        for idx in wrong_idx:
            misclassified_images.append(inputs[idx].cpu())
            misclassified_true.append(class_names[labels[idx].item()])
            misclassified_preds.append(class_names[preds[idx].item()])

            if len(misclassified_images) >= 10:
                break
        if len(misclassified_images) >= 10:
            break

print("Misclassified samples")

# Plotting the images
fig = plt.figure(figsize=(15, 6))
for i in range(10):
    ax = fig.add_subplot(2, 5, i+1, xticks=[], yticks=[])

    img = misclassified_images[i].numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    ax.imshow(img)
    ax.set_title(f"True: {misclassified_true[i]}\nPred: {misclassified_preds[i]}", color="red", fontweight='bold')

plt.tight_layout()
plt.show()